# 06. French's answer key: momentum and variance sorts of the universe

**What this notebook does.** Each month it ranks the universe's members on two signals known at the start of the month, momentum and the variance of daily returns, cuts them into five groups of equal size, and computes the return of the highest group minus the lowest, each stock in a group weighted by its market value. It sets these spreads beside the same spreads computed by Ken French within the largest stocks of the US market, and judges expectation E3 on the two comparisons. The same sorts in ten groups are reported beside them and not judged.

**Why it comes here.** French's data library publishes these spreads, built from the same CRSP files by a different hand and with different rules for which stocks count as large. If the universe is the set of large US stocks it claims to be, and its returns and weights are right, its spreads should move with French's month by month. This is the data layer's last check from outside CRSP's own index file, and the first that uses daily returns.

**Terms used in this notebook.**

- *universe*: the stocks in CRSP's list of index members on the last trading day of the month before.
- *verdict*: the result of an expectation, a test fixed in advance in `src/slt/constants.py`, met or not met, printed in the same words either way.
- *market capitalisation*: price times shares outstanding, at the end of the month before.
- *value-weighted*: weighted by market capitalisation, each stock's weight being its capitalisation divided by the sum over the stocks held. *Cap-weighted* means the same.
- *benchmark*: the value-weighted universe, as notebook 04 built it, with delisting returns applied.
- *momentum*: the return compounded over the eleven months from month t−12 to month t−2, leaving out month t−1, French's definition. A member enters the momentum sort for month t if it has a price at the end of month t−13, a return for month t−2 and a market capitalisation at the end of t−1, French's conditions. A missing return between t−12 and t−3 is skipped in the compounding and counted.
- *variance*: the variance of a member's daily returns over the 60 trading days ending on the last trading day of month t−1, on at least 20 returns, French's definition and minimum.
- *quintile* and *decile*: the five, or ten, groups of equal numbers of names, to within one, from lowest signal to highest. Ties are broken by CRSP's permanent number for the security, the PERMNO.
- *spread*: the value-weighted return of the highest quintile minus that of the lowest, in the month after the sort. Each group's return is taken over its members with a return, delisting returns applied, as the benchmark's is.
- *French's largest size quintile*: the stocks above the 80th percentile of the market capitalisations of NYSE stocks, in French's 25 portfolios formed on size and on a second signal. French's spread is his highest-signal portfolio minus his lowest within that quintile, value-weighted.
- *NYSE breakpoints*: French's quintiles are cut at percentiles of NYSE stocks alone, and then applied to all US stocks, so his groups are not of equal size.
- *vintage*: the version of CRSP's database a file was built from. French prints it on the first line of each file, and the extraction here holds CRSP to December 2024.
- *provenance*: the record of what a result was built from, here each French file's address, download time and SHA-256 fingerprint.
- *correlation*: the correlation coefficient of the two spreads over the months both exist.
- *standard error*: the standard deviation of the monthly difference between the two spreads divided by the square root of the number of months, the precision of their mean difference. E3 is judged on it. The *Newey-West standard error*, which allows the difference in one month to be correlated with the difference in the next twelve, is printed beside it with no verdict. *Autocorrelation* is that correlation of a series with its own past.
- *basis point*: a hundredth of a percentage point. A basis point a month in the mean difference is about 0.12 percentage points a year.

**What I expect to see, and where it comes from.**

- *E3*, for the momentum sort and for the variance sort: correlation at least 0.80 between the universe's quintile spread and French's, and a mean difference within two standard errors of zero, over the months both exist. The levels date from 6 September 2026 and were fixed before any data was read. French's largest size quintile is set by NYSE breakpoints and holds between about 100 and 530 names a month, fewer than the universe's 500 in almost every month from April 1957 and more than its 90 before, so the two sets overlap without coinciding and a correlation near one is not expected. Below 0.80 the universe is not the large-cap universe it claims to be. The universe goes from about 90 names to about 500 in April 1957, so the momentum comparison is also printed and committed for January 1927 to March 1957 and April 1957 to December 2024, with no verdict. E3's verdict is on the whole window.
- *The months compared*: momentum from January 1927, the first month with a price thirteen months back in the monthly stock file and a value in French's file, to December 2024. Variance from April 1990, the first month with 60 trading days of the daily stock file before it, to December 2024.
- *The members left out*: from structural counts made on 11 November 2026, before this notebook was written, about 2,500 of 440,000 member-months from January 1927 have no price at t−13 and about 100 have no return at t−2, and they are left out of the momentum sort. About 550 are sorted with a skipped month. In the variance sort 76 of 209,020 member-months from April 1990 have fewer than 20 daily returns and are left out.
- *Vintages*: French builds his files on CRSP's database of September 2026. Revisions between that vintage and the extraction's are part of any gap.

Running time: about five minutes, most of it reading the daily stock file. French's four files are downloaded once into `data/french/`, which git ignores.

In [ ]:
import json
import pathlib
import sys
import time

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from slt import constants as C
from slt import extraction as X

root = X.data_root()             # stops with one sentence when the data root is not set
manifest = X.check_manifest(root)   # stops when the manifest on disk is not the one the constants fix
print("manifest SHA-256", C.MANIFEST_SHA256[:12], "| files", manifest["counts"]["files"],
      "| rows", f"{manifest['counts']['rows']:,}")

In [ ]:
from slt import french as FR
from slt import layer as L
from slt import sorts as Q
from slt import validation as V

## 1. French's spreads

Each file is downloaded if it is not already in `data/french/`, and its provenance record is kept. From each of the two 25-portfolio files the notebook reads the block of value-weighted monthly returns and the block of the number of firms in each portfolio, cut at December 2024. French's spreads stay in memory: no French series is committed by month.

In [ ]:
french, french_firms, records = FR.spreads()
for r in records:
    print(r["filename"], "|", r["vintage_line"], "| SHA-256", r["zip_sha256"][:12], "| downloaded now:", r["fetched_this_run"])
for sort in C.SORTS:
    s = french[sort].dropna()
    print(f"{sort}: {s.index.min()} to {s.index.max()}, {len(s)} months with a value")
print("names in French's largest size quintile, mean by year:")
print(french_firms.loc[[y for y in (1927, 1957, 1958, 1980, 2000, 2024) if y in french_firms.index]].round(0).to_string())

## 2. The momentum sort

The universe and its monthly panel are rebuilt from the files as in notebooks 02 and 03. The signal is computed for every member-month from January 1927 from the monthly stock file, and the member-months that enter the sort are those with a signal and a market capitalisation.

In [ ]:
t0 = time.time()
universe = L.load_universe(root)
data = L.load_panel(root, universe)
panel = data["panel"]
wide = Q.monthly_wide(data["monthly"])
signals = {"momentum": Q.momentum(universe["ulong"], wide)}
entered = {"momentum": Q.entering(signals["momentum"], panel)}
counts = {"momentum": Q.counts_by_year(signals["momentum"], entered["momentum"], "momentum")}
c = counts["momentum"]
print(f"{time.time() - t0:.0f} s | member-months {c['member_months'].sum():,} | sorted {c['sorted'].sum():,}")
print(c.drop(columns="year").sum().to_dict())

## 3. The variance sort

The daily stock file is read a year at a time from 1990, for the stocks that are members in any month from April 1990, and held as one matrix of daily returns in memory. Each member's variance for month t uses its own returns over the 60 trading days before the month, whether or not it was a member on those days.

In [ ]:
t0 = time.time()
first_v = pd.Period(C.VARIANCE_SORT_START, "M")
ul = universe["ulong"]
permnos = ul.loc[ul["month"] >= first_v, "permno"].unique()
daily, calendar = L.daily_matrix(root, permnos, L.daily_years(), progress=True)
signals["variance"] = Q.variance(ul, daily, universe["ends"])
entered["variance"] = Q.entering(signals["variance"], panel)
counts["variance"] = Q.counts_by_year(signals["variance"], entered["variance"], "variance")
del daily
c = counts["variance"]
print(f"{time.time() - t0:.0f} s | member-months {c['member_months'].sum():,} | sorted {c['sorted'].sum():,}")
print(c.drop(columns="year").sum().to_dict())

## 4. The spreads against French's: E3

For each sort, the quintile spread and the decile spread of the universe by month, compared with French's spread over the months both exist. E3 is judged on the quintiles. The table by year is printed for every tenth year and committed in full.

In [ ]:
results, by_year, verdicts = {}, [], {}
for sort in C.SORTS:
    q5 = Q.spread(entered[sort], C.SORT_QUANTILES)
    q10 = Q.spread(entered[sort], C.SORT_QUANTILES_REPORTED)
    r5, r10 = Q.compare(q5, french[sort]), Q.compare(q10, french[sort])
    judged = Q.judge_e3(r5)
    results[sort] = {"quintiles": {**r5, **judged}, "deciles_no_verdict": r10}
    y5, y10 = Q.compare_by_year(q5, french[sort]), Q.compare_by_year(q10, french[sort])
    y = y5.merge(y10[["year", "correlation", "mean_universe_spread", "mean_difference", "standard_error"]],
                 on="year", suffixes=("_quintiles", "_deciles"))
    y = y.rename(columns={"months": "months", "mean_french_spread": "mean_french_spread"})
    names = Q.names_by_quantile(entered[sort], C.SORT_QUANTILES).rename("universe_names_per_quintile_mean")
    y = y.merge(names, left_on="year", right_index=True, how="left")
    y = y.merge(french_firms[sort].rename("french_names_largest_size_quintile_mean"), left_on="year", right_index=True, how="left")
    by_year.append(y.assign(sort=sort))
    print(V.verdict_line(f"E3, {sort}", judged["met"],
          f"Over {r5['months']} months from {r5['first_month']} to {r5['last_month']}, correlation {r5['correlation']:.3f} "
          f"against at least {C.E3_CORRELATION_MIN:.2f}, and a mean difference of {r5['mean_difference'] * 1e4:.1f} basis points "
          f"a month on a standard error of {r5['standard_error'] * 1e4:.1f}, {abs(r5['difference_in_standard_errors']):.2f} standard "
          f"errors against at most {C.E3_MEAN_DIFF_SE:.0f}"))
    print(f"   deciles, no verdict: correlation {r10['correlation']:.3f}, mean difference {r10['mean_difference'] * 1e4:.1f} "
          f"basis points on {r10['standard_error'] * 1e4:.1f} | Newey-West standard error of the quintiles' difference, "
          f"no verdict: {r5['standard_error_newey_west'] * 1e4:.1f}")
    verdicts[f"E3_{sort}"] = results[sort]["quintiles"]
    if sort == "momentum":
        results[sort]["eras_no_verdict"] = {}
        for a, b in C.MOMENTUM_ERAS:
            era = Q.compare(q5.loc[a:b], french[sort].loc[a:b])
            results[sort]["eras_no_verdict"][f"{a}_to_{b}"] = era
            print(f"   {a} to {b}, no verdict: {era['months']} months, correlation {era['correlation']:.3f}, mean difference "
                  f"{era['mean_difference'] * 1e4:.1f} basis points on a standard error of {era['standard_error'] * 1e4:.1f}")
by_year = pd.concat(by_year, ignore_index=True)
cols = ["sort", "year"] + [c for c in by_year.columns if c not in ("sort", "year")]
by_year = by_year[cols]
print(by_year[by_year["year"] % 10 == 0].round(4).to_string(index=False))

## 5. What is committed

Five files go into `results/20260928/`, each an aggregate with no identifier of a security and no series by month: the comparison by year for each sort, the counts of member-months by year and reason, a summary with the two comparisons over the window, the verdicts of milestone 2 (E3 here, E4's first clause added by notebook 07), and the provenance record of French's files.

In [ ]:
count_table = pd.concat([counts[s].assign(sort=s) for s in C.SORTS], ignore_index=True)
count_table = count_table[["sort"] + [c for c in count_table.columns if c != "sort"]].fillna(0)
summary = {
    "written_by": "notebooks/06_answer_key.ipynb", "manifest_sha256": C.MANIFEST_SHA256,
    "french_vintages": {r["key"]: r["vintage_line"] for r in records},
    "quantiles_judged": C.SORT_QUANTILES, "quantiles_reported": C.SORT_QUANTILES_REPORTED,
    "standard_error": C.E3_STANDARD_ERROR, "newey_west_lags_reported": C.E3_NEWEY_WEST_LAGS_REPORTED,
    "french_columns": {s: {"top": t, "bottom": b, "file": FR.C.FRENCH_FILES[k]} for s, (k, t, b) in C.E3_FRENCH_SPREADS.items()},
    "comparisons": results,
    "member_months": {s: {k: int(v) for k, v in counts[s].drop(columns=["year"]).sum().items()} for s in C.SORTS},
}
path = ROOT / C.RESULTS_DIR / "verdicts_milestone2.json"
record = json.loads(path.read_text()) if path.exists() else {}
record.update({k: {"met": v["met"], "correlation": v["correlation"], "mean_difference": v["mean_difference"],
                   "standard_error": v["standard_error"], "months": v["months"]} for k, v in verdicts.items()})
written = [X.write_results("answer_key_by_year.csv", frame=by_year),
           X.write_results("answer_key_counts_by_year.csv", frame=count_table),
           X.write_results("answer_key_summary.json", record=summary),
           X.write_results("verdicts_milestone2.json", record=record),
           FR.write_provenance(records)]
for p in written:
    print("written:", p.relative_to(ROOT))

## What was learned, and what could be wrong

What the notebook establishes: whether the universe's momentum and variance spreads move with French's large-stock spreads month by month and agree with them on average, judged as E3, with the deciles and the counts beside them.

What could still be wrong. French's quintiles are cut at NYSE percentiles and the universe's at its own, so a gap in level can come from the different groups as well as from an error. French's files are built on a later vintage of CRSP than the extraction, and revisions are part of any gap. A missing return between t−12 and t−3 is skipped here, where French's rule admits it only as a missing price, and the counts show how often that happens. And a correlation can be high while one spread runs above the other, which is why E3 judges the mean difference as well.